# PaMCP E8 — Single-tool retrieval campaign

This notebook runs E8 on two separate tracks: ToolRet (heterogeneous retrieval with graded qrels) and the acquired MCPToolBench++ subset (single resolved primary target only).

**Protocol boundaries**
- No pooling of ToolRet and MCPToolBench++ metrics.
- Discovery sees queries and the full candidate corpus only. Evaluator qrels remain separate.
- MCPToolBench++ primary-target metrics are the main result; source-annotated similar_tools are evaluated only as a separate sensitivity analysis.
- MCP-Atlas trajectories are not exhaustive acceptable-tool-set gold. Its E8 diagnostic remains blocked until a validated identity/coverage mapping exists.
- B1/B1m are blocked because their source files are not in this repository snapshot.
- This experiment does not measure execution success, composition validity, observed effects or security.

The runner saves rankings per task, macro metrics, paired CIs/tests, Holm-adjusted p-values, candidate recall for cross-encoder reranking, plots, hashes, cache metadata and a ZIP. It is resume-safe when the same run ID, data, model revisions, device and parameters are reused.

In [ ]:
import os, sys, json, subprocess, pathlib, time, platform
REPO = "https://github.com/msellamiTN/taas-provenance-discovery.git"
BRANCH = "research/pamcp-multibenchmark-v1"
REPO_DIR = pathlib.Path("/content/taas-provenance-discovery")

if REPO_DIR.exists() and (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout; remove it before cloning.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Branch:", BRANCH)
print("Commit:", REPO_COMMIT)

# Colab already ships with PyTorch on most runtimes. The model package remains pinned.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e",
    ".[test,experiment,benchmark-hf]", "sentence-transformers==5.1.2"
], check=True)

print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_e8_retrieval_campaign.py",
    "tests/test_toolret_adapter.py",
    "tests/test_mcp_atlas_adapter.py",
    "tests/test_mcpverse_adapter.py",
    "tests/test_evaluation_metrics.py",
    "tests/test_declared_effects_are_not_observations.py",
], check=True)

## 1. Acquire the pinned MCPToolBench++ subset

The source script fetches LiveMCPBench and MCPToolBench++ at the immutable commits in data/raw/SOURCES.txt. It reconstructs the local registry and the MCPToolBench++ task files. The E8 runner selects only single-primary-target examples for this track and records all exclusions.

In [ ]:
subprocess.run(["bash", "scripts/fetch_data.sh"], check=True)
sources = pathlib.Path("data/raw/SOURCES.txt").read_text(encoding="utf-8")
print(sources)
mcp_files = sorted(pathlib.Path("data/raw/mcptoolbenchpp").glob("*.json"))
assert mcp_files, "MCPToolBench++ raw JSON files were not acquired."
print("MCPToolBench++ category files:", [p.name for p in mcp_files])

## 2. Acquire ToolRet at its pinned revisions

ToolRet queries and tool corpus live in separate Hugging Face repositories. The candidate corpus is loaded independently from qrels; the adapter rejects qrels that refer to absent candidates. The revisions below are the previously recorded immutable snapshots.

In [ ]:
from huggingface_hub import HfApi
from taas.experiments.toolret_adapter import adapt_toolret_hf

TOOLRET_QUERY_REVISION = "b8c76ad3349ff17497b6bdb28bb5b8f61a0f6445"
TOOLRET_TOOL_REVISION = "e06c38c75612b6536bd959e08cdd345894aba6a7"

api = HfApi()
query_info = api.dataset_info("mangopy/ToolRet-Queries", revision=TOOLRET_QUERY_REVISION)
tool_info = api.dataset_info("mangopy/ToolRet-Tools", revision=TOOLRET_TOOL_REVISION)
assert query_info.sha == TOOLRET_QUERY_REVISION, query_info.sha
assert tool_info.sha == TOOLRET_TOOL_REVISION, tool_info.sha

toolret_manifest = adapt_toolret_hf(
    "data/processed/toolret",
    query_revision=TOOLRET_QUERY_REVISION,
    tool_revision=TOOLRET_TOOL_REVISION,
)
print(json.dumps(toolret_manifest["counts"], indent=2))
print("Data quality:", json.dumps(toolret_manifest.get("data_quality", {}), indent=2))
print("ToolRet query revision:", query_info.sha)
print("ToolRet tool revision:", tool_info.sha)

## 3. Check the two E8 tracks

The MCPToolBench++ corpus size should match the current repository subset (87 tools across 10 servers). The number of single-target tasks is computed by the runner after removing multiple-target, ambiguous and unresolved cases. ToolRet counts are read from its acquisition manifest; no historical aggregate is substituted.

In [ ]:
toolret_counts = toolret_manifest["counts"]
print("ToolRet:", json.dumps(toolret_counts, indent=2))

from taas.data.registries import load_mcptoolbenchpp
mcp_tools, mcp_tasks = load_mcptoolbenchpp(pathlib.Path("data/raw"))
print("MCPToolBench++ tools:", len(mcp_tools))
print("MCPToolBench++ servers:", len({t.server for t in mcp_tools}))
print("MCPToolBench++ task records:", len(mcp_tasks))
assert len(mcp_tools) == 87, f"Expected pinned repository subset of 87 tools; found {len(mcp_tools)}"
assert len({t.server for t in mcp_tools}) == 10, "Unexpected MCPToolBench++ server count."

## 4. Pin the retrieval model revisions

E5 uses query/passage prefixes. Cross-encoder reranking is limited to BM25 top-100 candidates and its candidate-recall ceiling is reported. This notebook resolves each model's current immutable Hugging Face commit once and passes those exact SHAs to the run; they are saved in the run manifest.

In [ ]:
import torch
from huggingface_hub import HfApi

E5_MODEL = "intfloat/e5-small-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

e5_info = api.model_info(E5_MODEL)
cross_info = api.model_info(CROSS_ENCODER_MODEL)
E5_REVISION = e5_info.sha
CROSS_ENCODER_REVISION = cross_info.sha
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("E5:", E5_MODEL, E5_REVISION)
print("Cross encoder:", CROSS_ENCODER_MODEL, CROSS_ENCODER_REVISION)
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: CPU execution is valid but may be slow for the full ToolRet track. A Colab GPU runtime is recommended.")

## 5. Execute E8

The runner's status is expected to be partial: B1/B1m, the MCP-Atlas diagnostic mapping, and some registered planning/adaptation baselines remain blocked. The available retrieval suite is still executed on each acquired track.

To resume an interrupted run in the same Colab filesystem, rerun this cell with the same E8_RUN_ID environment value. The runner validates source/model/config fingerprints and skips only matching completed baseline caches. Do not reuse a run ID after changing any input or model revision.

In [ ]:
OUTPUT_ROOT = pathlib.Path("/content/e8-results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = os.environ.get("E8_RUN_ID")
if not RUN_ID:
    RUN_ID = "E8-colab-" + time.strftime("%Y%m%dT%H%M%S", time.gmtime())
    os.environ["E8_RUN_ID"] = RUN_ID

RUN_DIR = OUTPUT_ROOT / RUN_ID
if (RUN_DIR / "MANIFEST.json").is_file():
    print("This run is already finalized and immutable; loading its manifest:", RUN_DIR)
else:
    command = [
        sys.executable, "experiments/e8_retrieval_campaign.py",
        "--root", str(REPO_DIR),
        "--output-root", str(OUTPUT_ROOT),
        "--toolret-dir", "data/processed/toolret",
        "--raw-root", "data/raw",
        "--e5-model", E5_MODEL,
        "--e5-revision", E5_REVISION,
        "--cross-encoder-model", CROSS_ENCODER_MODEL,
        "--cross-encoder-revision", CROSS_ENCODER_REVISION,
        "--device", DEVICE,
        "--top-k", "10",
        "--seed", "20261105",
        "--run-id", RUN_ID,
    ]
    print("Run ID:", RUN_ID)
    print("Command:", " ".join(command))
    subprocess.run(command, check=True)

manifest_path = RUN_DIR / "MANIFEST.json"
assert manifest_path.is_file(), f"Run manifest not found: {manifest_path}"
e8_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({
    "run_id": e8_manifest["run_id"],
    "status": e8_manifest["status"],
    "execution_scope_status": e8_manifest["execution_scope_status"],
    "tracks": {
        name: {
            "tasks": item["task_count_discovery"],
            "tools": item["tool_count_candidates"],
            "baseline_ids": item["baseline_ids_executed"],
            "cache_hits": item["cache_hits"],
        }
        for name, item in e8_manifest["tracks"].items()
    },
    "blocked_or_not_comparable": e8_manifest["blocked_or_not_comparable"],
}, indent=2))

## 6. Inspect metrics and plots

The ranking files preserve each task's top-10 list. The per-task CSVs and paired-test JSON files retain denominators, effect estimates, bootstrap intervals, raw p-values and Holm-adjusted p-values. MCPToolBench++ also has a separate sensitivity report that includes annotated similar_tools.

In [ ]:
for track_id, info in e8_manifest["tracks"].items():
    print("\nTRACK:", track_id)
    metrics_path = RUN_DIR / info["metrics_file"]
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    for baseline, entry in metrics.items():
        overall = entry["metrics"]["overall"]["metrics"]
        print(
            baseline,
            "Hit@1=", overall["1"]["hit@1"]["value"],
            "Hit@5=", overall["5"]["hit@5"]["value"],
            "MRR@10=", overall["10"]["mrr@10"]["value"],
            "macro_Hit@1=", entry["macro_hit@1"]["value"],
        )
    if info.get("similar_tools_sensitivity_metrics"):
        print("Similar-tools sensitivity:", info["similar_tools_sensitivity_metrics"])
    print("Plots:", info["plots"])

## 7. Export the immutable run bundle

The ZIP contains the manifest, input fingerprints, cached rankings, per-task metric files, paired tests, plots and hash inventory. Keep the ZIP with the exact run ID and model revisions when reporting results.

In [ ]:
archive_path = OUTPUT_ROOT / f"{RUN_ID}.zip"
assert archive_path.is_file(), f"Run archive missing: {archive_path}"
print("Run ID:", RUN_ID)
print("Manifest:", manifest_path)
print("Archive:", archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Google Colab. Retrieve the archive from:", archive_path)